# Sudoku Reloaded

Notebook pensado para ejecutarse en local desde la carpeta "cuadernos"
La foto debe estar bien orientada (números legibles).

### Realizo importaciones y la configuración de rutas

In [ ]:
import cv2 
import time
import numpy as np  
import matplotlib.pyplot as plt  
import copy                   
from pathlib import Path      
from ultralytics import YOLO  
from tensorflow import keras

# modelo de detección de sudokus en imágenes
RUTA_YOLO       = Path('../modelos/yolo.pt') 
# CNN entrenada con dígitos de fuentes impresas          
RUTA_MODELO_OCR = Path('../modelos/modelo_ocr.keras')  
# foto del sudoku a resolver
RUTA_IMAGEN     = Path('../img_pruebas/024.png')       
print('✓ Imports OK')

# compruebo que cada fichero existe antes de continuar; falla rápido si falta alguno
print(f'  YOLO:      {RUTA_YOLO}  ({"existe" if RUTA_YOLO.exists() else "NO ENCONTRADO"})')
print(f'  CNN OCR:   {RUTA_MODELO_OCR}  ({"existe" if RUTA_MODELO_OCR.exists() else "NO ENCONTRADO"})')
print(f'  Imagen:    {RUTA_IMAGEN}  ({"existe" if RUTA_IMAGEN.exists() else "NO ENCONTRADA"})')

### Carga de modelos

In [ ]:
# cargo el modelo YOLO desde el fichero .pt
modelo_yolo = YOLO(str(RUTA_YOLO))

# cargo la CNN de detección de números (clasifica imágenes de 28×28 en dígitos 0-9 (entrenada con fuentes impresas))
modelo_ocr  = keras.models.load_model(str(RUTA_MODELO_OCR))

print('✓ YOLO cargado')
print('✓ CNN OCR (MNIST) cargada')

# imprime la arquitectura de la CNN: capas, shapes de salida y nº de parámetros
modelo_ocr.summary()  

### Carga de imagen y detección del recuadro con YOLO

In [ ]:
# leo la imagen
imagen_original = cv2.imread(str(RUTA_IMAGEN))  
# fallo si no existe
assert imagen_original is not None, f'No se pudo cargar {RUTA_IMAGEN}'  
print(f'📷 {RUTA_IMAGEN.name}  ({imagen_original.shape[1]}×{imagen_original.shape[0]} px)') 

# ejecuto YOLO sobre la imagen 
results = modelo_yolo(imagen_original, verbose=False)

# si YOLO no detectó ningún objeto, informo y paro
if len(results[0].boxes) == 0:  
    print('❌ No se detectó ningún recuadro de sudoku')
else:
    # results[0].boxes.xyxy → tensor PyTorch con coordenadas [x1,y1,x2,y2] de cada detección
    # .cpu() mueve el tensor a CPU (por si estaba en GPU); .numpy() lo convierte a array numpy
    boxes = results[0].boxes.xyxy.cpu().numpy()

    # calculo el área de cada bounding box: ancho = x2-x1, alto = y2-y1
    areas = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])

    # me quedo con el índice de la caja más grande → suele ser el tablero completo
    idx   = int(np.argmax(areas))

    # extraigo las 4 coordenadas de la caja ganadora y las convierto a int para indexar píxeles
    x1, y1, x2, y2 = map(int, boxes[idx])

    # confianza de la detección: valor entre 0 y 1 que indica cuán seguro está YOLO
    conf = float(results[0].boxes.conf[idx])

    # recorto la región del tablero de la imagen original usando slicing numpy [filas, columnas]
    img_recortada = imagen_original[y1:y2, x1:x2]

    # hago una copia del original para no modificarlo al dibujar la caja
    img_bbox = imagen_original.copy()

    # dibujo el rectángulo de detección en verde (BGR: 0,200,0) con grosor 4 px
    cv2.rectangle(img_bbox, (x1,y1), (x2,y2), (0,200,0), 4)

    # creo una figura con dos paneles en horizontal
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    # panel izquierdo: imagen original con la caja de detección
    # cv2 usa BGR; matplotlib espera RGB → convierto con cvtColor
    axes[0].imshow(cv2.cvtColor(img_bbox, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'🎯 YOLO detecta ({conf:.1%})')  # confianza formateada como porcentaje
    axes[0].axis('off')  # oculto los ejes x/y (no son útiles para imágenes)

    # panel derecho: el recorte del tablero
    axes[1].imshow(cv2.cvtColor(img_recortada, cv2.COLOR_BGR2RGB))
    axes[1].set_title('✂️ Recorte')
    axes[1].axis('off')

    plt.tight_layout()  # ajusta márgenes para que los paneles no se solapen
    plt.show()          # renderizo la figura en el notebook
    print(f'✓ Recuadro: ({x1},{y1}) → ({x2},{y2})')  # coordenadas de la caja detectada

### Corrijo la perspectiva

In [ ]:
def ordenar_esquinas(pts):
    pts  = pts.reshape(4, 2).astype('float32')   # normalizo shape a (4,2) por si viene en otro formato
    rect = np.zeros((4, 2), dtype='float32')      # array destino para las 4 esquinas ordenadas

    s      = pts.sum(axis=1)          # suma x+y de cada punto; TL la tiene mínima, BR la máxima
    rect[0] = pts[np.argmin(s)]       # top-left:     menor suma x+y
    rect[2] = pts[np.argmax(s)]       # bottom-right: mayor suma x+y

    d      = np.diff(pts, axis=1).flatten()  # diferencia x-y de cada punto; TR la mín, BL la máx
    rect[1] = pts[np.argmin(d)]       # top-right:    menor diferencia x-y
    rect[3] = pts[np.argmax(d)]       # bottom-left:  mayor diferencia x-y
    return rect                       # devuelvo las esquinas en orden [TL, TR, BR, BL]


# detecta la cuadrícula INTERIOR (líneas negras) ignorando bordes de color
# corrige la perspectiva a un cuadrado de 450x450
def corregir_perspectiva(img):
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # convierto a escala de grises para simplificar
    blur = cv2.GaussianBlur(gris, (5, 5), 0)      # suavizo con kernel 5×5 para reducir ruido puntual

    # umbral adaptativo local: cada píxel se umbraliza según su vecindario de 11×11
    # THRESH_BINARY_INV → los píxeles oscuros (líneas) quedan en BLANCO (255)
    # constante 4 se resta a la media local para afinar el umbral
    thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 11, 4)

    kernel = np.ones((3, 3), np.uint8)            # kernel cuadrado de 3×3 para morfología
    thresh = cv2.dilate(thresh, kernel, iterations=1)  # dilato 1 vez para cerrar pequeños huecos en las líneas

    # busco contornos externos en la imagen umbralizada (RETR_EXTERNAL: solo los más externos)
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:  # si no hay ningún contorno, devuelvo la imagen sin modificar
        print('⚠️ No se encontraron contornos, usando imagen sin corregir')
        return img

    # ordeno los contornos de mayor a menor área para probar primero el más grande
    contornos = sorted(contornos, key=cv2.contourArea, reverse=True)
    img_area  = img.shape[0] * img.shape[1]  # área total de la imagen en píxeles

    for c in contornos[:8]:  # reviso solo los 8 contornos más grandes para no perder tiempo
        if cv2.contourArea(c) < img_area * 0.15:  # descarto si ocupa menos del 15% del área
            continue                                # la cuadrícula debería ocupar casi todo el recorte

        peri   = cv2.arcLength(c, True)            # perímetro del contorno (True = cerrado)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)  # simplifico con tolerancia del 2% del perímetro

        if len(approx) == 4:  # caso ideal: la aproximación dio exactamente 4 vértices (cuadrilátero)
            rect = ordenar_esquinas(approx.reshape(4, 2))  # ordeno los 4 puntos [TL,TR,BR,BL]
        else:
            # fallback: el rectángulo rotado mínimo siempre da 4 esquinas, aunque approx dé más
            box  = cv2.boxPoints(cv2.minAreaRect(c))  # 4 esquinas del rectángulo mínimo
            rect = ordenar_esquinas(box)               # las ordeno igual que el caso ideal

        lado = 450  # tamaño fijo del cuadrado de destino en píxeles
        # defino las 4 esquinas destino: TL=(0,0), TR=(450,0), BR=(450,450), BL=(0,450)
        dst  = np.array([[0,0],[lado,0],[lado,lado],[0,lado]], dtype='float32')

        # calculo la matriz de transformación homográfica 3×3 que mapea rect → dst
        M    = cv2.getPerspectiveTransform(rect, dst)

        # aplico la transformación a todos los píxeles de la imagen → resultado 450×450 cuadrado
        return cv2.warpPerspective(img, M, (lado, lado))

    print('⚠️ No se encontró cuadrícula clara, usando imagen sin corregir')
    return img  # si ningún contorno superó los filtros, devuelvo sin corregir


img_corregida = corregir_perspectiva(img_recortada)  # aplico la corrección de perspectiva

fig, axes = plt.subplots(1, 2, figsize=(10, 5))  # figura con dos paneles
axes[0].imshow(cv2.cvtColor(img_recortada,  cv2.COLOR_BGR2RGB))  # recorte original (torcido)
axes[0].set_title('Antes')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_corregida, cv2.COLOR_BGR2RGB))   # tablero ya cuadrado
axes[1].set_title('Después (perspectiva corregida)')
axes[1].axis('off')
plt.tight_layout()
plt.show()
print('✓ Perspectiva corregida')

### Divido en 81 celdas la imagen recortada

In [ ]:
# borra las líneas de la cuadrícula de toda la imagen a la vez 
# así las celdas quedan limpias y la CNN no confunde líneas con trazos del dígito
def eliminar_lineas(warped):
    """Borra las líneas de la cuadrícula de toda la imagen a la vez con morfología.
    Así las celdas quedan limpias y la CNN no confunde líneas con trazos del dígito."""
    gris    = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)  # paso a grises: trabajo con 1 canal

    # umbralizo: píxeles oscuros (líneas + dígitos) → BLANCO; fondo claro → NEGRO
    binaria = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY_INV, 11, 5)

    # kernel horizontal: 1 fila × (ancho/9) columnas ≈ 50 px de ancho
    # MORPH_OPEN con este kernel solo conserva rasgos horizontales de esa longitud → líneas H
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (warped.shape[1]//9, 1))
    h_lines  = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, h_kernel)  # máscara de líneas horizontales

    # kernel vertical: (alto/9) filas × 1 columna ≈ 50 px de alto
    # MORPH_OPEN con este kernel solo conserva rasgos verticales de esa longitud → líneas V
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, warped.shape[0]//9))
    v_lines  = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, v_kernel)  # máscara de líneas verticales

    # uno las dos máscaras con suma saturada (cv2.add) → máscara de TODA la cuadrícula
    # dilato 1 vez con kernel 3×3 para ensanchar la máscara y cubrir bordes de líneas
    lineas    = cv2.dilate(cv2.add(h_lines, v_lines), np.ones((3,3),np.uint8), iterations=1)

    resultado = gris.copy()          # parto de una copia de la imagen en grises
    resultado[lineas > 0] = 255      # donde la máscara es blanca, relleno con blanco → líneas borradas
    return resultado                 # devuelvo imagen en grises sin líneas de cuadrícula


# elimino las líneas de la cuadrícula ANTES de dividir en celdas
img_sin_lineas = eliminar_lineas(img_corregida)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))  # figura comparativa con/sin líneas
axes[0].imshow(cv2.cvtColor(img_corregida, cv2.COLOR_BGR2RGB))  # imagen con líneas
axes[0].set_title('Con líneas')
axes[0].axis('off')
axes[1].imshow(img_sin_lineas, cmap='gray')  # imagen en grises sin líneas (ya no es BGR)
axes[1].set_title('Sin líneas (lo que se divide)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# segmentación en 81 celdas
alto, ancho = img_sin_lineas.shape[:2]  # 450, 450 px (shape[:2] ignora el canal si existiera)
celda_alto  = alto  // 9               # altura de cada celda: 450÷9 = 50 px
celda_ancho = ancho // 9               # anchura de cada celda: 450÷9 = 50 px
MARGEN = 6  # píxeles que recortamos en cada borde de celda para eliminar restos de líneas

celdas = []  # lista 9×9 que contendrá cada celda como array numpy
for fila in range(9):       # itero por las 9 filas del tablero
    fila_celdas = []        # acumulo las 9 celdas de esta fila
    for col in range(9):    # itero por las 9 columnas del tablero
        y1c = fila * celda_alto  + MARGEN          # fila superior de la celda + margen de entrada
        y2c = y1c  + celda_alto  - MARGEN * 2      # fila inferior: resta margen sup e inf
        x1c = col  * celda_ancho + MARGEN          # columna izquierda + margen
        x2c = x1c  + celda_ancho - MARGEN * 2      # columna derecha: resta margen izq y der
        fila_celdas.append(img_sin_lineas[y1c:y2c, x1c:x2c])  # recorto la celda con slicing
    celdas.append(fila_celdas)  # añado la fila completa a la lista de celdas

fig, axes = plt.subplots(9, 9, figsize=(9, 9))  # rejilla de 9×9 subplots, uno por celda
for fila in range(9):
    for col in range(9):
        axes[fila][col].imshow(celdas[fila][col], cmap='gray')  # muestro cada celda en grises
        axes[fila][col].axis('off')  # oculto ejes para visualización limpia
plt.suptitle('81 celdas sin líneas', fontsize=12)
plt.tight_layout()
plt.show()
print('✓ 81 celdas generadas (sin líneas de cuadrícula)')

### Lectura de dígitos con la CNN anteriormente entrenada

In [ ]:
def celda_tiene_digito(celda):
    """Detecta si una celda tiene dígito por contraste centro vs esquinas."""
    # si la celda es BGR la convierto a grises; si ya es 2D (grises) la uso directamente
    gris = celda if len(celda.shape)==2 else cv2.cvtColor(celda, cv2.COLOR_BGR2GRAY)
    gris = cv2.resize(gris.astype(np.float32), (48, 48))  # reescalo a 48×48 para análisis uniforme
    e = 8  # tamaño del bloque de esquina: 8×8 px en cada una de las 4 esquinas

    # concateno los 4 bloques de esquina en un solo vector para estimar el color del fondo
    # TL: gris[:e,:e], TR: gris[:e,-e:], BL: gris[-e:,:e], BR: gris[-e:,-e:]
    esquinas = np.concatenate([gris[:e,:e].ravel(), gris[:e,-e:].ravel(),
                                gris[-e:,:e].ravel(), gris[-e:,-e:].ravel()])
    fondo  = np.median(esquinas)  # mediana de las esquinas → estimación robusta del fondo (suele ser ~255)
    centro = gris[12:36, 12:36]   # región central 24×24 donde debe aparecer el dígito

    # si más del 3% de los píxeles centrales son >40 puntos más oscuros que el fondo → hay dígito
    return (np.sum(centro < fondo - 40) / centro.size) > 0.03


def preprocesar_para_cnn(celda):
    """Celda gris (sin líneas) → 28x28 dígito NEGRO sobre fondo BLANCO (como el entrenamiento)."""
    # aseguro que trabajo en grises
    gris = celda if len(celda.shape)==2 else cv2.cvtColor(celda, cv2.COLOR_BGR2GRAY)

    # reescalo a 28×28 px: tamaño de entrada estándar de la CNN; INTER_AREA es mejor para reducir
    gris = cv2.resize(gris, (28, 28), interpolation=cv2.INTER_AREA)

    # CLAHE: ecualización adaptativa de histograma con límite de contraste
    # mejora el contraste localmente (tiles 4×4) sin amplificar el ruido de fondo
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    gris  = clahe.apply(gris)  # aplico la ecualización local

    # THRESH_OTSU calcula automáticamente el umbral óptimo entre fondo y trazo
    # SIN _INV: fondo blanco → 255, trazo negro → 0 (igual que el dataset de entrenamiento)
    _, binaria = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binaria.astype('float32') / 255.0  # normalizo a [0.0, 1.0] para la CNN


def predecir_celda(celda, modelo):
    """Clasifica una celda: 0=vacía, 1-9=dígito."""
    if not celda_tiene_digito(celda):  # heurística rápida: si no hay dígito, evito la inferencia
        return 0  # celda vacía

    # preproceso y reformo a tensor (1, 28, 28, 1): batch=1, alto=28, ancho=28, canales=1
    img  = preprocesar_para_cnn(celda).reshape(1, 28, 28, 1)

    # inferencia: pred es el vector softmax de 10 probabilidades (clases 0-9)
    pred = modelo.predict(img, verbose=0)[0]  # [0] elimina la dimensión de batch

    # tomo la clase con mayor probabilidad
    clase = int(np.argmax(pred))

    # la clase 0 no representa el cero del sudoku (es una etiqueta de entrenamiento)
    # si argmax devuelve 0, tomo la segunda mejor clase dentro del rango 1-9
    if clase == 0:
        clase = int(np.argmax(pred[1:])) + 1  # pred[1:] son las probabilidades de 1-9; +1 corrige el índice
    return clase  # devuelvo el dígito reconocido (1-9)


sudoku = []  # lista que acumulará las 9 filas de la matriz numérica
for fila in range(9):    # recorro las 9 filas
    fila_nums = []       # acumulo los 9 valores de esta fila
    for col in range(9): # recorro las 9 columnas
        fila_nums.append(predecir_celda(celdas[fila][col], modelo_ocr))  # clasifico la celda
    sudoku.append(fila_nums)  # añado la fila a la matriz

# cuento dígitos detectados y huecos para estadística
n = sum(1 for f in sudoku for x in f if x != 0)
print(f'✓ {n} dígitos detectados · {81-n} huecos\n')

print('Sudoku detectado:')
print('─' * 25)
for i, fila in enumerate(sudoku):
    if i % 3 == 0 and i != 0: print('─' * 25)  # separador de región cada 3 filas
    s = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0: s += '│ '    # separador vertical de región cada 3 columnas
        s += f'{num if num != 0 else "·"} '     # dígito o · para celdas vacías
    print(s)

### Verificación: imagen + matriz lado a lado

In [ ]:
fig = plt.figure(figsize=(15, 7))  # figura grande para mostrar imagen y tablero cómodamente

# panel izquierdo: imagen corregida (referencia visual)
ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(cv2.cvtColor(img_corregida, cv2.COLOR_BGR2RGB))  # convierto BGR→RGB para matplotlib
ax1.set_title('Cuadrícula detectada', fontsize=14, fontweight='bold')
ax1.axis('off')

# panel derecho: tablero numérico dibujado a mano con matplotlib
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_xlim(0, 9)       # el eje x va de 0 a 9 (una unidad por columna)
ax2.set_ylim(0, 9)       # el eje y va de 0 a 9 (una unidad por fila)
ax2.set_aspect('equal')  # fuerzo celdas cuadradas (relación aspecto 1:1)
ax2.axis('off')          # oculto ejes de coordenadas
ax2.set_title('Matriz detectada (CNN OCR)', fontsize=13, fontweight='bold')

for i in range(9):       # itero por filas
    for j in range(9):   # itero por columnas
        d = sudoku[i][j] # valor de la celda (0=vacío, 1-9=dígito)
        # celda con dígito → fondo verde claro; celda vacía → fondo gris claro
        fondo = '#d4edda' if d != 0 else '#f7f7f7'
        # dibujo el rectángulo de la celda; 8-i invierte el eje Y (fila 0 arriba)
        ax2.add_patch(plt.Rectangle((j, 8-i), 1, 1, facecolor=fondo, edgecolor='#bbb', lw=0.5))
        if d != 0:  # solo escribo el número si la celda no está vacía
            ax2.text(j+0.5, 8-i+0.5, str(d), ha='center', va='center',
                     fontsize=17, fontweight='bold', color='#222')  # centrado en la celda

for k in range(10):  # dibujo 10 líneas de cuadrícula (0..9 incluye ambos extremos)
    lw = 2.5 if k % 3 == 0 else 0.5  # líneas más gruesas cada 3 celdas (bordes de región 3×3)
    ax2.axhline(k, color='#333', lw=lw)  # línea horizontal
    ax2.axvline(k, color='#333', lw=lw)  # línea vertical

plt.tight_layout()
plt.show()

### Resolución con backtracking

In [ ]:
def es_valido(t, fila, col, num):
    # compruebo que num no esté ya en la misma fila (t[fila] es la lista de 9 valores)
    if num in t[fila]: return False

    # compruebo que num no esté ya en la misma columna (extraigo la columna con list comprehension)
    if num in [t[i][col] for i in range(9)]: return False

    # calculo el inicio de la región 3×3 a la que pertenece la celda (fila,col)
    bf, bc = (fila//3)*3, (col//3)*3  # bf = primera fila del bloque, bc = primera columna
    for i in range(bf, bf+3):         # recorro las 3 filas del bloque
        for j in range(bc, bc+3):     # recorro las 3 columnas del bloque
            if t[i][j] == num: return False  # num ya está en la región 3×3

    return True  # num no viola ninguna de las tres restricciones → es válido


def backtrack(t):
    for i in range(9):        # recorro filas
        for j in range(9):    # recorro columnas
            if t[i][j] == 0:  # encontré una celda vacía; aquí aplico la recursión
                for num in range(1, 10):         # pruebo cada dígito del 1 al 9
                    if es_valido(t, i, j, num):  # solo continúo si el dígito es válido
                        t[i][j] = num            # coloco el dígito provisionalmente
                        if backtrack(t): return True  # llamo recursivamente; si resuelve, propago True
                        t[i][j] = 0              # el dígito no llevó a solución → lo deshago (backtrack)
                return False  # ningún dígito del 1-9 fue válido → callejón sin salida, retrocedo
    return True  # no quedan celdas vacías → el tablero está completamente resuelto


tablero  = copy.deepcopy(sudoku)  # clono el tablero original para no perder las pistas (los 0s)
resuelto = backtrack(tablero)     # intento resolver in-place sobre la copia

if resuelto:
    print('✓ Sudoku resuelto (backtracking):\n')
else:
    print('❌ No se pudo resolver — puede haber errores de lectura en el OCR\n')

print('─' * 25)
for i, fila in enumerate(tablero):
    if i % 3 == 0 and i != 0: print('─' * 25)  # separador de región
    s = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0: s += '│ '  # separador vertical de región
        marca = ' ' if sudoku[i][j] != 0 else '*'  # * indica que el solver rellenó ese hueco
        s += f'{num}{marca}'
    print(s)
print('(* = rellenado por backtracking)')

### Resolución con la red neuronal convolucional

Se carga el modelo entrenado con un millón de sudokus del CSV y lo aplico directamente sobre los 81 valores detectados

In [ ]:
# cargo el modelo de resolución
RUTA_SOLVER = Path('../modelos/modelo_sudoku_mejor.keras')  

# cargo el modelo 
modelo_solver = keras.models.load_model(str(RUTA_SOLVER))
print('✓ modelo solver cargado')
modelo_solver.summary()  # muestro la arquitectura de la red

# preparo el input
puzzle_flat = np.array([v for fila in sudoku for v in fila], dtype='float32')  # aplano la matriz 9×9 a vector de 81 elementos
puzzle_norm = puzzle_flat / 9.0  # normalizo: 1→0.111, 9→1.0, 0→0.0 (huecos)
puzzle_input = puzzle_norm.reshape(1, 81)  # añado dimensión de batch: (1, 81)

# predicción de los 81 dígitos 
t0 = time.time() # anoto el tiempo de inicio
pred_raw = modelo_solver.predict(puzzle_input, verbose=0)[0]   # elimina la dimensión de batch: shape (81, 9)
# pred_raw[i] es el vector softmax de 9 probabilidades para la celda i (clases 0-8, que representan dígitos 1-9)
pred_clases = np.argmax(pred_raw, axis=1)   # tomo la clase con mayor probabilidad para cada celda → shape (81,)
digitos_cnn = pred_clases + 1  # sumo 1 porque el modelo usa clases 0-8 pero los dígitos del sudoku son 1-9
t_cnn = time.time() - t0  # calculo el tiempo transcurrido en segundos

# donde el puzzle tiene un dígito (≠0) mantengo el original; donde hay hueco (=0) uso la predicción
sol_cnn = np.where(puzzle_flat != 0, puzzle_flat.astype(int), digitos_cnn)  # np.where actúa celda a celda
tablero_cnn = sol_cnn.reshape(9, 9).tolist()                                 # convierto a lista 9×9 para visualizar igual que backtracking

# estadísticas de la predicción 
huecos_mask = puzzle_flat == 0   # máscara booleana: True donde había hueco
n_huecos    = huecos_mask.sum()  # cuántas celdas tenía que predecir
print(f'✓ cnn solver ejecutada en {t_cnn*1000:.1f} ms')
print(f'  huecos a rellenar: {n_huecos}')
print(f'  dígitos predichos: {digitos_cnn[huecos_mask]}')

# imprimo el tablero resuelto por la cnn 
print('\ntablero resuelto por cnn solver:')
print('─' * 25)
for i, fila in enumerate(tablero_cnn):
    if i % 3 == 0 and i != 0: print('─' * 25)  # separador de región 3×3
    s = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0: s += '│ '    # separador vertical de región
        marca = ' ' if sudoku[i][j] != 0 else '~'  # ~ indica celda rellenada por la cnn
        s += f'{num}{marca}'
    print(s)
print('(~ = predicho por la red neuronal)')

### Comparación entre CNN y backtracking
Medimos cuántas celdas acierta cada método y cuánto tarda. La red neuronal es casi instantánea pero puede cometer errores;
el backtracking es 100% fiable pero más lento.

In [ ]:
import time  # ya importado antes, pero lo repito por si se ejecuta esta celda sola

#  recupero los tiempos del backtracking para poder comparar 
# vuelvo a medir el backtracking sobre una copia limpia para tener su tiempo real
tablero_bt2  = copy.deepcopy(sudoku)   # hago una copia profunda del puzzle original para no modificarlo
t0_bt        = time.time()             # anoto el tiempo de inicio del backtracking
backtrack(tablero_bt2)                 # resuelvo in-place; modifica tablero_bt2
t_bt         = time.time() - t0_bt    # tiempo total del backtracking en segundos

# comparación celda a celda en los huecos 
# solo me interesan las celdas que eran huecos (donde cada método tuvo que trabajar)
# tablero (backtracking) y tablero_cnn son listas 9×9; las convierto a arrays para operar
arr_bt  = np.array(tablero)     # solución del backtracking → array (9,9)
arr_cnn = np.array(tablero_cnn) # solución de la cnn        → array (9,9)
arr_orig = np.array(sudoku)     # puzzle original con huecos → array (9,9)

# máscara de los huecos en el puzzle original (0 = hueco)
mask_huecos = arr_orig == 0     # array booleano (9,9): True donde había hueco
n_huecos    = mask_huecos.sum() # número total de huecos que tenía el puzzle

# el backtracking siempre da la solución correcta si el puzzle es válido
# lo uso como referencia de verdad para evaluar la cnn
aciertos_cnn = np.sum(arr_cnn[mask_huecos] == arr_bt[mask_huecos])  # celdas que la cnn acertó
pct_cnn      = aciertos_cnn / n_huecos * 100                         # porcentaje de acierto
perfecta     = aciertos_cnn == n_huecos                              # True si la cnn acertó todo

#  tabla de resultados 
print('\n' + '═'*52)
print(f'  {"método":<24}{"celdas correctas":<18}{"tiempo"}')
print('─'*52)
print(f'  {"cnn solver (red neuronal)":<24}{f"{aciertos_cnn}/{n_huecos} ({pct_cnn:.1f}%)":<18}{t_cnn*1000:>7.1f} ms')
print(f'  {"backtracking":<24}{f"{n_huecos}/{n_huecos} (100.0%)":<18}{t_bt*1000:>7.1f} ms')
print('═'*52)

if perfecta:
    print('\n✅ la cnn resolvió el puzzle perfectamente sin necesitar backtracking')
else:
    errores = n_huecos - aciertos_cnn  # número de celdas que la cnn falló
    print(f'\n⚠️  la cnn falló en {errores} celda(s) → el backtracking corrige y garantiza el 100%')
    print('   esto demuestra por qué combinamos ambos métodos en el pipeline')

print('\n→ conclusión para la presentación:')
print('  · la red neuronal es casi instantánea pero no garantiza solución perfecta')
print('  · el backtracking es 100% fiable pero explora el espacio de búsqueda')
print('  · combinados: rapidez de la red + garantía del backtracking')

### Solución visual: detectado + solución CNN + backtracking

In [ ]:
# muestro tres paneles: puzzle detectado / resuelto por cnn / resuelto por backtracking
fig = plt.figure(figsize=(20, 7))  # figura ancha para tres tableros lado a lado

# defino los tres conjuntos de datos y sus títulos para iterar
paneles = [
    (sudoku,      'detectado (ocr)',          None),          # puzzle original con huecos
    (tablero_cnn, 'cnn solver',               'cnn'),         # solución de la red neuronal
    (tablero,     'backtracking',             'bt'),          # solución del backtracking
]

for idx, (datos, titulo, metodo) in enumerate(paneles):  # itero sobre los tres paneles
    ax = fig.add_subplot(1, 3, idx+1)      # creo el subplot en la posición correcta (1 fila, 3 columnas)
    ax.set_xlim(0, 9)                      # el eje x va de 0 a 9 (una unidad por columna)
    ax.set_ylim(0, 9)                      # el eje y va de 0 a 9 (una unidad por fila)
    ax.set_aspect('equal')                 # fuerzo celdas cuadradas
    ax.axis('off')                         # oculto los ejes de coordenadas
    ax.set_title(titulo, fontsize=13, fontweight='bold')  # título del panel

    for i in range(9):      # itero por las 9 filas
        for j in range(9):  # itero por las 9 columnas
            d     = datos[i][j]       # valor de la celda en este tablero
            orig  = sudoku[i][j]      # valor original del puzzle (0 si era hueco)
            es_hueco = orig == 0      # True si esta celda era un hueco a rellenar

            # lógica de color según el panel y si la celda era hueco o pista
            if metodo == 'cnn' and es_hueco:       # celda rellenada por la cnn
                # comparo con backtracking para saber si acertó o falló
                acertada = (d == tablero[i][j])    # True si la cnn coincide con backtracking
                if acertada:
                    color_txt, fondo = '#16a34a', '#dcfce7'  # verde: la cnn acertó
                else:
                    color_txt, fondo = '#dc2626', '#fee2e2'  # rojo: la cnn falló
            elif metodo == 'bt' and es_hueco:      # celda rellenada por backtracking
                color_txt, fondo = '#2563eb', '#eff6ff'      # azul: solución garantizada
            elif d != 0:                           # pista original (igual en todos los paneles)
                color_txt, fondo = '#222222', '#ffffff'      # negro sobre blanco
            else:                                  # hueco vacío (solo en panel 'detectado')
                color_txt, fondo = '#222222', '#f3f4f6'      # gris claro para vacíos

            # dibujo el rectángulo de la celda con el color asignado
            # 8-i invierte el eje y para que la fila 0 quede arriba (como en un sudoku real)
            ax.add_patch(plt.Rectangle((j, 8-i), 1, 1,
                         facecolor=fondo, edgecolor='#9ca3af', lw=0.5))

            if d != 0:  # solo escribo el número si la celda tiene valor
                ax.text(j+0.5, 8-i+0.5, str(d),
                        ha='center', va='center',
                        fontsize=15, fontweight='bold', color=color_txt)

    # dibujo la cuadrícula del tablero
    for k in range(10):                                   # 10 líneas cubren los 9 intervalos
        lw = 2.5 if k % 3 == 0 else 0.5                  # línea gruesa cada 3 celdas (borde de región)
        ax.axhline(k, color='#374151', lw=lw)             # línea horizontal
        ax.axvline(k, color='#374151', lw=lw)             # línea vertical

# leyenda de colores debajo de los paneles
leyenda = (
    'verde = cnn acertó   │   rojo = cnn falló   │   '
    'azul = backtracking (100% fiable)   │   negro = pista original'
)
fig.text(0.5, 0.01, leyenda, ha='center', fontsize=10, color='#374151')  # texto centrado al pie

plt.tight_layout(rect=[0, 0.04, 1, 1])  # dejo espacio abajo para la leyenda
plt.show()